# Stage 1 — Svensson Curve Fit, Micro Factor & PCA

- Fit Svensson (1994) extended Nelson-Siegel at every minute × auction.
- Compute `micro_factor` = OTR yield − fitted 30y (Hu-Pan-Wang illiquidity proxy).
- Fit intraday PCA basis (train-only; re-done inside every CV fold in Stage 6).

**Reads:** `data/cache/intraday_evtclk.parquet`  
**Writes:** `data/cache/intraday_curved.parquet`

Refs: Gürkaynak-Sack-Wright (2007); Hu-Pan-Wang (2013).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import CACHE_DIR, MATURITIES, N_PCA
from src.curve import (
    svensson,
    fit_curve_one_minute,
    add_micro_factor,
    fit_pca_basis,
    add_pca_projections,
)

intraday_df = pd.read_parquet(CACHE_DIR / 'intraday_evtclk.parquet')
print(f'Loaded: {len(intraday_df):,} rows  |  {intraday_df["auction_id"].nunique()} auctions')

## 1a. Svensson fits + micro factor

~105k curve fits. Uses warm-starts per minute within each auction.  
Results cached to `intraday_curved.parquet` — subsequent runs are instant.

In [ ]:
intraday_df = add_micro_factor(intraday_df, use_cache=True)  # set False to refit
print('micro_factor sample:')
intraday_df[['auction_id', 'timestamp_et', 'otr_30y_yield', 'fitted_30y',
             'micro_factor', 'noise_xsec']].head()

## 1b. Micro factor diagnostics

In [ ]:
print(intraday_df['micro_factor'].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
intraday_df['micro_factor'].hist(bins=80, ax=axes[0])
axes[0].set_title('micro_factor distribution')
axes[0].set_xlabel('bps')

intraday_df['noise_xsec'].hist(bins=80, ax=axes[1])
axes[1].set_title('Cross-section noise (Svensson RMSE)')
axes[1].set_xlabel('bps')

plt.tight_layout()
plt.show()

## 1c. Intraday PCA basis (full-sample preview only)

> **Note:** This full-sample fit is for inspection only.  
> Inside CV (Stage 6), `fit_pca_basis` is called on training rows only.

In [ ]:
pca_intra = fit_pca_basis(intraday_df)   # train-only inside CV folds
intraday_df = add_pca_projections(intraday_df, pca_intra)

print('PCA components shape:', pca_intra.components_.shape)
print('Columns added:', [c for c in intraday_df.columns if c.startswith('pc')])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
taus = MATURITIES
labels = ['PC1 (Level)', 'PC2 (Slope)', 'PC3 (Curvature)']
for i, (ax, lbl) in enumerate(zip(axes, labels)):
    ax.plot(taus, pca_intra.components_[i], marker='o')
    ax.set_title(lbl)
    ax.set_xlabel('Maturity (yr)')
    ax.set_xticks(taus)
plt.tight_layout()
plt.show()

## 1d. Persist

In [ ]:
intraday_df.to_parquet(CACHE_DIR / 'intraday_stage1.parquet', index=False)
print('Saved → cache/intraday_stage1.parquet')